In [5]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os

import duckdb
import pandas as pd
import pyarrow.parquet as pq

drive.mount("/content/drive", force_remount=False)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
TEMP_DIR = Path("/content/duckdb_tmp")

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {max(1, min(os.cpu_count() or 4, 8))}")
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

parquet = pq.ParquetFile(PARQUET_PATH)
metadata = parquet.metadata

print("File :", PARQUET_PATH)
print("Size :", f"{PARQUET_PATH.stat().st_size / 1024**2:.2f} MB")
print("Rows :", f"{metadata.num_rows:,}")
print("Cols :", metadata.num_columns)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Size : 469.49 MB
Rows : 3,469
Cols : 12


In [6]:
LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS = 4
MIN_SEGMENTS = 90
SESSIONS_PER_LANGUAGE = 5

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

con.execute(
    f"""
    CREATE OR REPLACE TABLE session_stats AS
    SELECT
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at,
        lang_detected AS language_code,
        lang_probability,
        list_count(
            list_filter(
                transcript_segments,
                segment ->
                    segment.words IS NOT NULL
                    AND len(segment.words) >= {MIN_WORDS}
            )
        ) AS segment_count
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    WHERE
        lang_detected IN ({language_sql})
        AND transcript_segments IS NOT NULL
        AND len(transcript_segments) >= {MIN_SEGMENTS}
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TABLE selected_sessions AS
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY language_code
                ORDER BY
                    segment_count ASC,
                    created_at DESC NULLS LAST,
                    gamesession_id DESC
            ) AS session_rank
        FROM session_stats
        WHERE segment_count >= {MIN_SEGMENTS}
    )
    SELECT
        language_code,
        session_rank,
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_probability,
        segment_count
    FROM ranked
    WHERE session_rank <= {SESSIONS_PER_LANGUAGE}
    """
)

selected_sessions = con.execute(
    """
    SELECT *
    FROM selected_sessions
    ORDER BY
        language_code,
        session_rank
    """
).df()

selected_sessions.insert(
    0,
    "language",
    selected_sessions["language_code"].map(LANGUAGES),
)

display(selected_sessions)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language,language_code,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,German,de,1,141268064,467269,Naraka,https://www.twitch.tv/videos/2854317619,gen10,2026-08-23 23:29:07,0.8291,98
1,German,de,2,141264210,399263,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2854157061,warfare4,2026-08-23 18:39:26,0.9072,107
2,German,de,3,141265651,687197,Escape from Tarkov,https://www.twitch.tv/videos/2854208944,gen10,2026-08-23 21:53:10,0.9199,114
3,German,de,4,141236998,833542,COD: Warzone3-2,https://www.twitch.tv/videos/2853489925,warfare2,2026-08-23 03:22:23,0.6826,115
4,German,de,5,141238626,812794,COD: Warzone3-2,https://www.twitch.tv/videos/2853620823,warfare2,2026-08-23 04:09:16,0.9678,118
5,English,en,1,141266766,395792,Phasmophobia,https://www.twitch.tv/videos/2854265768,gen10,2026-08-23 22:19:05,0.9463,90
6,English,en,2,141245486,773042,COD: Modern Warfare III,https://www.youtube.com/watch?v=UAUU-ciTr08,warfare3,2026-08-23 12:30:09,0.9277,90
7,English,en,3,141250470,615244,Helldivers 2,https://www.twitch.tv/videos/2853894951,gen4,2026-08-23 09:36:17,0.9976,90
8,English,en,4,141210406,466455,COD: Warzone3-2,https://www.twitch.tv/videos/2852833829,warfare2,2026-08-23 00:10:17,0.8442,90
9,English,en,5,141236416,100090,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2853520407,warfare4,2026-08-23 09:08:50,0.6890,91


In [7]:
con.execute(
    f'''
    CREATE OR REPLACE TABLE selected_segments AS
    WITH source AS (
        SELECT
            p.gamesession_id,
            p.lang_detected AS language_code,
            p.transcript_segments
        FROM read_parquet('{PARQUET_PATH.as_posix()}') AS p
        INNER JOIN selected_sessions AS s
            ON p.gamesession_id = s.gamesession_id
            AND p.lang_detected = s.language_code
    ),
    exploded AS (
        SELECT
            gamesession_id,
            language_code,
            generate_subscripts(transcript_segments, 1) AS segment_index,
            UNNEST(transcript_segments) AS segment
        FROM source
    )
    SELECT
        gamesession_id,
        language_code,
        segment_index,
        TRIM(segment.text) AS segment_text,
        len(segment.words) AS word_count
    FROM exploded
    WHERE
        segment.words IS NOT NULL
        AND len(segment.words) >= {MIN_WORDS}
        AND segment.text IS NOT NULL
        AND TRIM(segment.text) <> ''
    '''
)

selected_segment_counts = con.execute(
    '''
    SELECT
        language_code,
        gamesession_id,
        COUNT(*) AS selected_segment_count,
        MIN(word_count) AS min_word_count
    FROM selected_segments
    GROUP BY
        language_code,
        gamesession_id
    ORDER BY
        language_code,
        gamesession_id
    '''
).df()

display(selected_segment_counts)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,language_code,gamesession_id,selected_segment_count,min_word_count
0,de,141236998,115,4
1,de,141238626,118,4
2,de,141264210,107,4
3,de,141265651,114,4
4,de,141268064,98,4
5,en,141210406,90,4
6,en,141236416,91,4
7,en,141245486,90,4
8,en,141250470,90,4
9,en,141266766,90,4


In [8]:
session_validation = con.execute(
    f"""
    WITH language_checks AS (
        SELECT
            language_code,
            COUNT(*) AS session_count,
            MIN(segment_count) AS min_segment_count,
            COUNT(*) = {SESSIONS_PER_LANGUAGE} AS has_five_sessions,
            MIN(segment_count) >= {MIN_SEGMENTS} AS all_sessions_have_min_segments
        FROM selected_sessions
        GROUP BY language_code
    ),
    word_checks AS (
        SELECT
            language_code,
            MIN(word_count) AS min_word_count,
            MIN(word_count) >= {MIN_WORDS} AS all_segments_have_min_words
        FROM selected_segments
        GROUP BY language_code
    ),
    order_checks AS (
        SELECT
            language_code,
            bool_and(
                next_segment_count IS NULL
                OR segment_count <= next_segment_count
            ) AS smallest_segment_count_first
        FROM (
            SELECT
                language_code,
                session_rank,
                segment_count,
                lead(segment_count) OVER (
                    PARTITION BY language_code
                    ORDER BY session_rank
                ) AS next_segment_count
            FROM selected_sessions
        )
        GROUP BY language_code
    )
    SELECT
        l.language_code,
        l.session_count,
        l.min_segment_count,
        w.min_word_count,
        l.has_five_sessions,
        l.all_sessions_have_min_segments,
        w.all_segments_have_min_words,
        o.smallest_segment_count_first,
        (
            l.has_five_sessions
            AND l.all_sessions_have_min_segments
            AND w.all_segments_have_min_words
            AND o.smallest_segment_count_first
        ) AS all_checks_passed
    FROM language_checks AS l
    INNER JOIN word_checks AS w USING (language_code)
    INNER JOIN order_checks AS o USING (language_code)
    ORDER BY language_code
    """
).df()

session_validation.insert(
    0,
    "language",
    session_validation["language_code"].map(LANGUAGES),
)

display(session_validation)

if len(session_validation) != len(LANGUAGES):
    raise ValueError("Validation did not cover all target languages")

if not session_validation["all_checks_passed"].all():
    raise ValueError("Validation failed")

print("All validation checks passed.")

,language,language_code,session_count,min_segment_count,min_word_count,has_five_sessions,all_sessions_have_min_segments,all_segments_have_min_words,smallest_segment_count_first,all_checks_passed
0,German,de,5,98,4,True,True,True,True,True
1,English,en,5,90,4,True,True,True,True,True
2,Spanish,es,5,94,4,True,True,True,True,True
3,French,fr,5,130,4,True,True,True,True,True
4,Portuguese,pt,5,91,4,True,True,True,True,True
5,Russian,ru,5,90,4,True,True,True,True,True


All validation checks passed.


In [10]:
!apt-get -qq update
!apt-get -qq install -y libicu-dev pkg-config build-essential

%pip -q install \
    aiohttp \
    fasttext \
    pycld2 \
    morfessor \
    PyICU \
    polyglot

import asyncio
import json
import re
import time
import urllib.request
from pathlib import Path

import aiohttp
import fasttext
import pandas as pd
from google.colab import userdata
from polyglot.detect import Detector

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is missing")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

MODELS = {
    "ox_alpha": "stealth/ox-alpha",
    "nemotron_3_5_lightning": "nvidia/nemotron-3.5-lightning:free",
    "glm_5_2": "z-ai/glm-5.2:free",
    "gemma_4_31b": "google/gemma-4-31b-it:free",
    "hy_mt2_30b_a3b": "tencent/hy-mt2-30b-a3b",
}

ALLOWED_LANGUAGES = {
    "en",
    "de",
    "fr",
    "pt",
    "es",
    "ru",
}

SEGMENTS_PER_SESSION = 5
BATCH_SIZE = 10
MAX_CONCURRENCY = 3
MAX_RETRIES = 5
REQUEST_TIMEOUT = 120

print("Polyglot:", Detector("This is a test.", quiet=True).language.code)
print("OpenRouter key: OK")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118422 files and directories currently installed.)
Removing r-base-dev (4.6.1-5.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libglib2.0-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libfontconfig-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.

Re

In [11]:
benchmark_segments = con.execute(
    f"""
    WITH ranked AS (
        SELECT
            s.language_code,
            s.gamesession_id,
            s.session_rank,
            g.segment_index,
            g.segment_text,
            g.word_count,
            row_number() OVER (
                PARTITION BY g.gamesession_id
                ORDER BY g.segment_index
            ) AS segment_position,
            count(*) OVER (
                PARTITION BY g.gamesession_id
            ) AS session_segment_count
        FROM selected_segments AS g
        INNER JOIN selected_sessions AS s
            ON g.gamesession_id = s.gamesession_id
            AND g.language_code = s.language_code
    ),
    targets AS (
        SELECT
            *,
            round(
                segment_position
                * ({SEGMENTS_PER_SESSION} + 1.0)
                / (session_segment_count + 1.0)
            ) AS bucket
        FROM ranked
    ),
    sampled AS (
        SELECT *
        FROM targets
        QUALIFY row_number() OVER (
            PARTITION BY
                gamesession_id,
                bucket
            ORDER BY
                abs(
                    segment_position
                    - bucket
                    * (session_segment_count + 1.0)
                    / ({SEGMENTS_PER_SESSION} + 1.0)
                ),
                segment_index
        ) = 1
    ),
    final_ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY gamesession_id
                ORDER BY segment_position
            ) AS sample_rank
        FROM sampled
        WHERE bucket BETWEEN 1 AND {SEGMENTS_PER_SESSION}
    )
    SELECT
        language_code AS dataset_language,
        gamesession_id,
        segment_index,
        segment_text
    FROM final_ranked
    WHERE sample_rank <= {SEGMENTS_PER_SESSION}
    ORDER BY
        dataset_language,
        session_rank,
        segment_index
    """
).df()

counts = (
    benchmark_segments
    .groupby(["dataset_language", "gamesession_id"])
    .size()
)

if not (counts == SEGMENTS_PER_SESSION).all():
    raise ValueError("Every session must contribute exactly 5 segments")

if len(benchmark_segments) != 150:
    raise ValueError(
        f"Expected 150 benchmark segments, found {len(benchmark_segments)}"
    )

display(benchmark_segments)

,dataset_language,gamesession_id,segment_index,segment_text
0,de,141268064,25,Alter. Einfach gleiche Stun.
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh."
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannende..."
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr."
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und ..."
...,...,...,...,...
145,ru,141262832,43,Поэтому я... в возврат кинул хуйню.
146,ru,141262832,73,еще далинка нужна очень сильно мне
147,ru,141262832,103,Хочу на Леончике карточку сыграть. Родировку. ...
148,ru,141262832,132,"вот меня палец есть, у него палец раскачу"


In [13]:
FASTTEXT_MODEL_PATH = Path("/content/lid.176.ftz")

if not FASTTEXT_MODEL_PATH.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
        FASTTEXT_MODEL_PATH,
    )

FASTTEXT_MODEL = fasttext.load_model(
    FASTTEXT_MODEL_PATH.as_posix()
)


def normalize_fasttext_text(text):
    if pd.isna(text):
        return ""

    return " ".join(
        str(text)
        .replace("\n", " ")
        .replace("\r", " ")
        .split()
    )


def predict_fasttext(text):
    text = normalize_fasttext_text(text)

    if not text:
        return "other"

    try:
        labels, probabilities = FASTTEXT_MODEL.predict(
            text,
            k=1,
            threshold=0.0,
        )

        if len(labels) == 0:
            return "other"

        language_code = labels[0].removeprefix("__label__")

        return (
            language_code
            if language_code in ALLOWED_LANGUAGES
            else "other"
        )

    except Exception:
        return "error"


benchmark_segments["fasttext_prediction"] = (
    benchmark_segments["segment_text"]
    .map(predict_fasttext)
)


def predict_polyglot(text):
    text = re.sub(r"\s+", " ", str(text)).strip()

    if not text:
        return "other"

    try:
        detector = Detector(
            text,
            quiet=True,
        )

        return normalize_prediction(
            detector.language.code
        )

    except Exception:
        return "other"


benchmark_segments["fasttext_prediction"] = (
    benchmark_segments["segment_text"]
    .map(predict_fasttext)
)

benchmark_segments["polyglot_prediction"] = (
    benchmark_segments["segment_text"]
    .map(predict_polyglot)
)

display(benchmark_segments)

,dataset_language,gamesession_id,segment_index,segment_text,fasttext_prediction,polyglot_prediction
0,de,141268064,25,Alter. Einfach gleiche Stun.,error,de
1,de,141268064,46,"Ja irgendwie schon, Alter, aber es macht weh.",error,de
2,de,141268064,63,"Ah, ja. Ja, voll. Ja, das sicher ein spannende...",error,de
3,de,141268064,87,"Legende eins, let´s go! Das guckt nicht mehr.",error,de
4,de,141268064,106,"Bro, ich hasse es. War das der Heng? Heng und ...",error,de
...,...,...,...,...,...,...
145,ru,141262832,43,Поэтому я... в возврат кинул хуйню.,error,ru
146,ru,141262832,73,еще далинка нужна очень сильно мне,error,ru
147,ru,141262832,103,Хочу на Леончике карточку сыграть. Родировку. ...,error,ru
148,ru,141262832,132,"вот меня палец есть, у него палец раскачу",error,other


In [14]:
SYSTEM_PROMPT = """
You are a language identification classifier.

Determine the primary spoken language represented by each transcript.

Allowed outputs:
en
de
fr
pt
es
ru
other

Return only the requested JSON object.
Do not translate.
Do not explain.
Do not correct the transcript.
Infer the language from the text exactly as provided.
""".strip()


def make_batches(frame):
    records = (
        frame
        .reset_index()
        .rename(columns={"index": "row_id"})
        [["row_id", "segment_text"]]
        .to_dict("records")
    )

    return [
        records[index:index + BATCH_SIZE]
        for index in range(0, len(records), BATCH_SIZE)
    ]


def make_prompt(batch):
    payload = [
        {
            "id": int(row["row_id"]),
            "text": row["segment_text"],
        }
        for row in batch
    ]

    return (
        "Classify every item.\n"
        'Return exactly: {"predictions":['
        '{"id":0,"language":"en"}'
        "]}\n\n"
        + json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        )
    )


def extract_json(content):
    content = str(content).strip()

    content = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        content,
        flags=re.IGNORECASE,
    )

    start = content.find("{")
    end = content.rfind("}")

    if start < 0 or end < start:
        raise ValueError("JSON object not found")

    return json.loads(
        content[start:end + 1]
    )


def parse_predictions(content, expected_ids):
    parsed = extract_json(content)
    predictions = parsed.get("predictions", [])

    results = {}

    for item in predictions:
        try:
            row_id = int(item["id"])
            language = normalize_prediction(
                item["language"]
            )
        except Exception:
            continue

        if row_id in expected_ids:
            results[row_id] = language

    return results


def parse_single_prediction(content):
    content = str(content).lower()

    matches = re.findall(
        r"\b(en|de|fr|pt|es|ru|other)\b",
        content,
    )

    if not matches:
        return "other"

    return matches[-1]


async def request_openrouter(
    session,
    semaphore,
    model,
    messages,
):
    payload = {
        "model": model,
        "temperature": 0,
        "max_tokens": 256,
        "messages": messages,
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(MAX_RETRIES):
        try:
            async with semaphore:
                async with session.post(
                    OPENROUTER_URL,
                    headers=headers,
                    json=payload,
                ) as response:
                    data = await response.json(
                        content_type=None
                    )

                    if response.status == 429:
                        retry_after = float(
                            response.headers.get(
                                "Retry-After",
                                2 ** attempt,
                            )
                        )
                        await asyncio.sleep(retry_after)
                        continue

                    if response.status >= 400:
                        message = (
                            data.get("error", {})
                            .get("message", str(data))
                        )

                        raise RuntimeError(
                            f"{model}: "
                            f"HTTP {response.status}: "
                            f"{message}"
                        )

                    return data[
                        "choices"
                    ][0]["message"]["content"]

        except Exception:
            if attempt + 1 == MAX_RETRIES:
                raise

            await asyncio.sleep(
                min(2 ** attempt, 16)
            )

    raise RuntimeError(model)

In [ ]:
async def classify_single(
    session,
    semaphore,
    model,
    row,
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Detect the language of this transcript.\n"
                "Return exactly one language code and nothing else.\n\n"
                f"{row['segment_text']}"
            ),
        },
    ]

    content = await request_openrouter(
        session,
        semaphore,
        model,
        messages,
    )

    return (
        int(row["row_id"]),
        parse_single_prediction(content),
    )


async def classify_batch(
    session,
    semaphore,
    model,
    batch,
):
    expected_ids = {
        int(row["row_id"])
        for row in batch
    }

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": make_prompt(batch),
        },
    ]

    try:
        content = await request_openrouter(
            session,
            semaphore,
            model,
            messages,
        )

        results = parse_predictions(
            content,
            expected_ids,
        )

    except Exception:
        results = {}

    missing = [
        row
        for row in batch
        if int(row["row_id"]) not in results
    ]

    if missing:
        fallback_results = await asyncio.gather(
            *[
                classify_single(
                    session,
                    semaphore,
                    model,
                    row,
                )
                for row in missing
            ]
        )

        results.update(
            dict(fallback_results)
        )

    return results


async def run_openrouter_models(frame):
    batches = make_batches(frame)

    semaphore = asyncio.Semaphore(
        MAX_CONCURRENCY
    )

    timeout = aiohttp.ClientTimeout(
        total=REQUEST_TIMEOUT
    )

    model_results = {
        name: {}
        for name in MODELS
    }

    async with aiohttp.ClientSession(
        timeout=timeout
    ) as session:
        for model_name, model_id in MODELS.items():
            print(f"Running {model_name}...")

            completed = await asyncio.gather(
                *[
                    classify_batch(
                        session,
                        semaphore,
                        model_id,
                        batch,
                    )
                    for batch in batches
                ]
            )

            for result in completed:
                model_results[
                    model_name
                ].update(result)

    output = frame.reset_index(drop=True).copy()

    for model_name in MODELS:
        output[
            f"{model_name}_prediction"
        ] = [
            model_results[model_name].get(
                row_id,
                "other",
            )
            for row_id in range(len(output))
        ]

    return output


benchmark_results = await run_openrouter_models(
    benchmark_segments
)

display(benchmark_results)

Running ox_alpha...
Running nemotron_3_5_lightning...
